# What Gets Captured

Understanding what data is captured and how it's stored is essential for effective data analysis.

## The Bluesky Document Model

Bluesky uses a **document-based data model** with four types of documents:

```{mermaid}
graph TD
    A[Start Document] --> B[Descriptor Document(s)]
    B --> C[Event Document 1]
    B --> D[Event Document 2]
    B --> E[Event Document ...]
    B --> F[Event Document N]
    C --> G[Stop Document]
    D --> G
    E --> G
    F --> G
```

### 1. Start Document (One per run)

Contains:
- Run UID (unique identifier)
- Timestamp
- Plan name and arguments
- Metadata (experiment_id, operator, purpose, etc.)
- Device configurations

### 2. Descriptor Document(s) (One or more per run)

Contains:
- Data schema (what fields are captured)
- Data keys and their properties (shape, dtype, units)
- Object names (detectors, motors, etc.)

### 3. Event Documents (Many per run)

Contains:
- Actual measurement data
- One event per scan point
- Readings from all devices
- Timestamps for each reading

### 4. Stop Document (One per run)

Contains:
- Exit status (success/fail)
- Final timestamp
- Total number of events

## Let's See Documents in Action

Run an experiment and examine the documents created:

In [ ]:
import sys
from pathlib import Path

parent_dir = Path.cwd().parent if 'jupyter_book_tutorial' in str(Path.cwd()) else Path.cwd()
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

from bluesky import RunEngine
from ophyd import Signal
from bluesky.plans import scan
from bluesky_config.devices import MockDetector
from bluesky_config.metadata import create_experiment_metadata

# We'll capture documents in a list to examine them
captured_docs = []

def document_collector(name, doc):
    """Callback to collect all documents."""
    captured_docs.append((name, doc))

# Setup
RE = RunEngine({})
RE.subscribe(document_collector)

motor = Signal(name='motor', value=0)
detector = MockDetector(name='det')

# Run a small scan
uid = RE(scan([detector], motor, 0, 5, 5))

print(f"✓ Scan complete. Captured {len(captured_docs)} documents.")

### Examine the Start Document

In [ ]:
import json

# Get start document
start_name, start_doc = captured_docs[0]

print(f"Document type: {start_name}")
print(f"\nKey fields in start document:")
print(json.dumps({
    'uid': start_doc['uid'][:16] + '...',
    'time': start_doc['time'],
    'plan_name': start_doc['plan_name'],
    'plan_args': start_doc.get('plan_args', {}),
    'scan_id': start_doc.get('scan_id'),
}, indent=2))

### Examine the Descriptor Document

In [ ]:
# Get descriptor document
desc_name, desc_doc = captured_docs[1]

print(f"Document type: {desc_name}")
print(f"\nData keys (what will be measured):")
for key, info in desc_doc['data_keys'].items():
    print(f"  - {key}: {info['dtype']} (shape: {info['shape']})")

### Examine Event Documents

In [ ]:
# Get event documents (skip start and descriptor)
events = [(name, doc) for name, doc in captured_docs[2:-1] if name == 'event']

print(f"Total events: {len(events)}")
print(f"\nFirst event data:")
first_event = events[0][1]
print(json.dumps({
    'seq_num': first_event.get('seq_num'),
    'data': first_event['data'],
    'timestamps': first_event['timestamps']
}, indent=2))

print(f"\nLast event data:")
last_event = events[-1][1]
print(json.dumps({
    'seq_num': last_event.get('seq_num'),
    'data': last_event['data'],
}, indent=2))

### Examine the Stop Document

In [ ]:
# Get stop document
stop_name, stop_doc = captured_docs[-1]

print(f"Document type: {stop_name}")
print(f"\nStop document:")
print(json.dumps({
    'run_start': stop_doc['run_start'][:16] + '...',
    'time': stop_doc['time'],
    'exit_status': stop_doc['exit_status'],
    'num_events': stop_doc.get('num_events', {})
}, indent=2))

## Storage Locations

Data is stored in two places:

### 1. Document Files (`data/documents/`)

- Human-readable JSON Lines format
- One file per run: `<uid>_documents.jsonl`
- Can be opened in text editor
- Permanently stored (survives environment restart)

Let's look at the actual file:

In [ ]:
# Check for document files
doc_dir = parent_dir / 'data' / 'documents'
doc_files = list(doc_dir.glob('*.jsonl'))

if doc_files:
    latest_file = max(doc_files, key=lambda p: p.stat().st_mtime)
    print(f"Latest document file: {latest_file.name}")
    print(f"Size: {latest_file.stat().st_size} bytes")
    print(f"\nFirst 5 lines:")
    
    with open(latest_file, 'r') as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            doc = json.loads(line)
            doc_type = list(doc.keys())[0]
            print(f"  Line {i+1}: {doc_type} document")
else:
    print("No document files found yet.")

### 2. DataBroker Catalog (Optional)

- Can use temp (in-memory), sqlite (persistent), or mongodb
- Configured in `config.py`
- Enables advanced querying and searching
- Can index by metadata

## What Data Gets Captured Per Event

During each scan point (event), the following are captured:

- **Motor/parameter positions**: Current value of scanned parameter
- **Detector readings**: All detector values at this position
- **Timestamps**: Exact time of measurement
- **Metadata**: Any custom metadata you add

Let's visualize the data flow:

In [ ]:
# Show all captured data as a table
import pandas as pd

# Extract data from events
data_rows = []
for name, doc in captured_docs:
    if name == 'event':
        row = doc['data'].copy()
        row['seq_num'] = doc.get('seq_num', 0)
        data_rows.append(row)

df = pd.DataFrame(data_rows)
df = df[['seq_num', 'motor', 'det_value']]  # Reorder columns

print("Captured data as table:")
print(df)

## Key Takeaways

```{important}
**Document Flow:**
1. **Start** - Run begins, metadata recorded
2. **Descriptor** - Define what will be measured
3. **Events** - One per data point (N events for N points)
4. **Stop** - Run ends, status recorded
```

```{tip}
**Storage:**
- Documents saved to `data/documents/<uid>_documents.jsonl`
- Also indexed in DataBroker catalog for easy retrieval
- Both storage methods provide the same data, just different access patterns
```

## Next Steps

Now that you understand what gets captured, learn how to verify your data was saved correctly:

→ **Chapter 4: [Verifying Data Capture](04_verification.ipynb)**